![image_1787419385294.png](./image_1787419385294.png "image_1787419385294.png")

# Contexto
En el sector asegurador, la detección temprana de posibles casos de fraude permite focalizar esfuerzos de investigación, proteger la sostenibilidad técnica del portafolio y mejorar la eficiencia operativa de los equipos de siniestros, auditoría y gestión del riesgo. Esta prueba busca evaluar la capacidad del candidato para transformar una necesidad de negocio en un problema analítico abordable, construir un modelo predictivo con datos reales o simulados, interpretar sus resultados y comunicar sus hallazgos de forma clara para audiencias técnicas y de negocio.


#Reto técnico
Construir una solución analítica para estimar la probabilidad de que un registro, siniestro, reclamación, transacción o caso del negocio asegurador corresponda a un posible fraude. El candidato deberá trabajar con una única matriz de datos suministrada por la compañía, cuya variable objetivo es FraudeS /N, y desarrollar un flujo completo de análisis y modelado, desde la exploración inicial hasta la sustentación de resultados.
Se dará puntos adicionales si la solución es desarrollada en Databricks Free Edition, en caso contrario deberá ser implementada en Python, Incluyendo Git para control de versiones con un repositorio organizado.

#Configuración del repositorio
Es importante tener un versionamiento del proyecto por lo que se vincula este notebook a un repositorio previamente creado y sincronizado con databricks

#Instalación de paquetes

In [0]:
#instalamos la librerías
#para lectura de datos
%pip install -q openpyxl
#para mixed nulls
%pip install -q deepchecks --upgrade
#para estadistica descriptiva
%pip install -q "pathspec<0.12"
%pip install -q scikit-build-core cmake ninja pybind11
%pip install -q --no-build-isolation "phik==0.12.5"
%pip install -q ydata_profiling
#para catboost
%pip install -q catboost

#Reinicio del entorno

In [0]:
%restart_python

#Importación de librerías

In [0]:
#Manejo de datos
import pandas as pd
import numpy as np

if not hasattr(np, "Inf"):
    np.Inf = np.inf

from sklearn.preprocessing import LabelEncoder
#librerías gráficas
import seaborn as sns
import matplotlib.pyplot as plt

#Reconocimiento de nulos
from deepchecks.tabular.checks import MixedNulls
#Validación cruzada
from sklearn.model_selection import KFold
#Modelación
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from joblib import dump
#métricas de evaluación
from sklearn.metrics import accuracy_score
from sklearn import metrics
#hiperparametrización
from sklearn.model_selection import GridSearchCV
#Correlacion
from scipy.stats import chi2_contingency
#Descripcion estadistica
from ydata_profiling import ProfileReport

#Lectura de los datos
Hacemos la lectura de los datos que ya previamente fueron subidos al catalog de databricks, adicionalmente miramos unos cuantos registros para entender mejor la estructura de los datos y el formato de las variables

In [0]:
Ruta_base = "/Volumes/workspace/prueba_tecnica/muestra_base_fraude/Muestra_Base_fraude.xlsx"
Base = pd.read_excel(Ruta_base) 
Base.head(5)

*   Contamos con una base de **12.776 filas** y **40 columnas** , esto puede cambiar a medida de que hagamos transformaciones y limpieza.

*   Podemos apreciar el **tipo de dato** y **cantidad de no nulos** en cada variable, esto último se debe analizar junto con **conocimiento de negocio** ya que en el caso de algunas variables puede ser **normal** el tener muchos nulos
*   Adicionalmente nos podemos dar una idea del **nivel de completitud** de cada variable, (esto no quere decir que sea el escenario final de completitud , ya que los varores **nulos** pueden estar en **diferentes formatos**, no todos necesariamente detectados por la función)

In [0]:
Base.info()

#Detección de registros duplicados
Primero inspeccionamos la base virgen en busca de registros repetidos y al parecer tenemos 329 reclamos Duplicados. (Puede que mas adelante cuando tenga mas entendimiento sobre la base pueda hacer combinaciones buscando otro tipo de duplicidades) 

In [0]:
##Detección de registros duplicados
Base.duplicated().value_counts()

Hacemos una primera limpieza de **registros duplicados** los cuales pueden ensuciar nuestros futuros modelos , nos quedamos con **12447 registros unicos** , con eso nos aseguramos que cada registro es una reclamacion unica.

In [0]:
Base_sin_duplicados = Base.drop_duplicates().reset_index(drop=True)

#Entendimiento de los datos
* Segun se puede ver a simple vista esta base de datos representa un conjunto de reclamaciones de seguros de vida (rentas, invalidez, incapacidades) hechas por clientes en diferentes ventanas de tiempo , cada registro trae todo el ciclo del siniestro "desde la vigencia de la póliza hasta el cierre" junto con la etiqueta de si fue detectado como fraude o no

* Con el fin de conocer mas a fondo la naturaleza de la base y asi poder descartar todas las variables que por definición son irrelevantes para la construccion del modelo , se crea un glosario con el entendimiento de cada variable

### Producto y canal de venta

- **Ramo**: código del ramo del seguro.
- **Ramo_Desc**: descripción del ramo (debe ser homóloga a *Ramo*).
- **Nombre_plan**: producto específico al que pertenece la reclamación.
- **Codigo_Canal_Comercial_Op** / **Nombre_Canal_Comercial**: código y nombre del canal por el que se vendió el seguro.
- **Amparo_Desc**: nombre del amparo (cobertura) afectado por la reclamación.

### Póliza y asegurado

- **Fecha_Primera_Vigencia_Cert**: fecha de primera vigencia del certificado individual (me interesa mas esta fecha por que es mas exacta que la master).
- **fecha_primera_vigencia_pol**: fecha de primera vigencia de la póliza máster. En seguros individuales debería coincidir con la del certificado.
- **IDENTIFICACION_asegurado**: número de identificación del asegurado.
- **SEXO_asegurado**: género del asegurado.
- **edad_ingreso_asegurado** / **edad_actual_asegurado**: edad al vincularse a la compañía vs. edad al momento de la reclamación (gran potencial para definir antiguedad).
- **vigencia_poliza**: número de vigencias/renovaciones de la póliza; funciona como proxy de antigüedad del cliente.
- **vigencia_certificado**: similar a *vigencia_poliza* pero a nivel de certificado.
- **FEXPEDICION**: todo indica que es la fecha de expedición de la póliza/certificado, no del reclamo. Coincide de cerca con las fechas de primera vigencia, y en 91% de los casos es anterior a *FSINIESTRO* — la póliza se expide antes de que ocurra el siniestro, como debería ser.

### Estructura comercial

- **CODSUC** / **SUCURSAL**: código y nombre de la oficina.
- **REGIONAL**: regional a la que pertenece la oficina.
- **AGENTE** / **CODAG**: nombre e identificador del agente o asociación que vendió el seguro.

### El siniestro en sí

- **CAUSASTRO** / **DESCAUSA**: id y nombre de la causa de la reclamación.
- **DIAGNOSTICO**: diagnóstico médico asociado.
- **FSINIESTRO**: fecha en que ocurrió el siniestro.
- **F_Notificacion**: fecha en que se notificó el siniestro a la compañía.
- **Fecha_Recepcion**: fecha de recepción formal de la reclamación.
- **Fecha_Apertura**: fecha de apertura del caso.
- **Fecha_Primer_Cierre_Siniestro**: fecha del primer cierre. 
- **Ind_Tipo_Atencion**: canal/modalidad de atención (interna vs. externa).
- **Ind_Pago_Automatico**: si el pago se hizo de forma automática (S/N).

### Montos y estado

- **Sum(Valor_Reservas_Inicial)**: reserva inicial constituida.
- **Sum(Valor_Reservas)**: reserva final/actual.
- **Sum(Valor_Pagos)**: valor efectivamente pagado.
- **estado**: estado actual de la reclamación.
- **Cobertura**: cobertura afectada — se cruza directamente con *Amparo_Desc*.
- **Tipo apertura**: medio por el que se atendió la reclamación, muy relacionada con *Ind_Tipo_Atencion*.

### Variable objetivo y reporte

- **Fraude S /N**: variable objetivo — si la reclamación fue determinada como fraude.
- **Periodo Reporte**: mes del reporte.
- **Fecha de reporte**: fecha completa del reporte del fraude.
- **Año**: año del reporte.

# Eliminacion de variables irrelevantes por definición

- **Ramo**: como id no tiene ningun valor descriptivo, ademas ya existe su version en texto llamada Ramo_Desc.
- **Codigo_Canal_Comercial_Op**: igual que Ramo, no aporta nada como codigo y tiene su homologo descriptivo en Nombre_Canal_Comercial.
- **CODSUC**: el id en si no tiene valor descriptivo, ya existe su homologo SUCURSAL con el nombre de la oficina.
- **CODAG**: comparte la misma descripcion que AGENTE, asi que se queda solo este ultimo.
- **CAUSASTRO**: es un id sin valor descriptivo, su homologo con descripcion es DESCAUSA.

 NOTA: a pesar que la variable IDENTIFICACION_asegurado tiene la misma naturaleza que estas variables , aun no la elimino por que a partir de ella puedo crear otras variables , ademas esta variable es fijo  (data leakage) debido a la identidad y cardinalidad

Eliminamos las variables anteriormente mencionadas, quedamos con 35 de las 40 variables originales

In [0]:
columnas_a_eliminar = ['Ramo', 'Codigo_Canal_Comercial_Op', 'CODSUC', 'CODAG', 'CAUSASTRO']

Base_sin_duplicados = Base_sin_duplicados.drop(columns=columnas_a_eliminar)

print(f"Columnas eliminadas: {columnas_a_eliminar}")
print(f"Dimensiones actuales: {Base_sin_duplicados.shape}")

# Eliminacion de variables redundantes por definición

- **fecha_primera_vigencia_pol**: comparte gran parte de sus valores con Fecha_Primera_Vigencia_Cert (un 68% de sus valores). Me quedo con esta ultima porque da mayor detalle del cliente al ser la vigencia del certificado y no de la poliza master.
- **vigencia_poliza**: comparte varios valores con vigencia_certificado (en un 76%), y al igual que en el caso anterior prefiero quedarme con la version de certificado.
- **F_Notificacion**, **Fecha_Recepcion**, **Fecha_Apertura**: estas tres fechas son casi identicas entre si. Me quedo con **F_Notificacion** porque el nombre es mas diciente y hace referencia directa a la fecha de aviso del siniestro.
- **Periodo Reporte**, **Fecha de reporte**, **Año**: estas tres hacen referencia a la misma fecha en distintos niveles de detalle, asi que me quedo unicamente con **Fecha de reporte** por ser la mas completa.

A continuacion mostramos las coincidencias entre variables mencionadas en el texto anteior, que evidencian su redundancia

In [0]:
coincidencia_vigencias = (Base['Fecha_Primera_Vigencia_Cert'] == Base['fecha_primera_vigencia_pol']).mean()

print(f"Porcentaje de coincidencia exacta entre ambas fechas: {coincidencia_vigencias:.2%}")

In [0]:
coincidencia_vigencia_num = (Base['vigencia_poliza'] == Base['vigencia_certificado']).mean()

print(f"Porcentaje de coincidencia exacta entre ambas vigencias: {coincidencia_vigencia_num:.2%}")

In [0]:
coincidencia_notif_recep = (Base['F_Notificacion'] == Base['Fecha_Recepcion']).mean()
coincidencia_notif_apert = (Base['F_Notificacion'] == Base['Fecha_Apertura']).mean()
coincidencia_recep_apert = (Base['Fecha_Recepcion'] == Base['Fecha_Apertura']).mean()
coincidencia_las_tres = ((Base['F_Notificacion'] == Base['Fecha_Recepcion']) & 
                          (Base['Fecha_Recepcion'] == Base['Fecha_Apertura'])).mean()

print(f"F_Notificacion == Fecha_Recepcion: {coincidencia_notif_recep:.2%}")
print(f"F_Notificacion == Fecha_Apertura: {coincidencia_notif_apert:.2%}")
print(f"Fecha_Recepcion == Fecha_Apertura: {coincidencia_recep_apert:.2%}")
print(f"Las tres coinciden al mismo tiempo: {coincidencia_las_tres:.2%}")

In [0]:
meses = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril',5:'Mayo',6:'Junio',7:'Julio',
         8:'Agosto',9:'Septiembre',10:'Octubre',11:'Noviembre',12:'Diciembre'}

Base['mes_de_fecha_reporte'] = Base['Fecha de reporte'].dt.month.map(meses)

coincidencia_mes = (Base['mes_de_fecha_reporte'] == Base['Periodo Reporte']).mean()
coincidencia_anio = (Base['Fecha de reporte'].dt.year == Base['Año']).mean()

print(f"Mes de Fecha de reporte == Periodo Reporte: {coincidencia_mes:.2%}")
print(f"Año de Fecha de reporte == Año: {coincidencia_anio:.2%}")

Por último eliminamos las variables consideradas redundantes por definición, quedando con 29 de las 40 variables originales

In [0]:
columnas_redundantes = [
    'fecha_primera_vigencia_pol',   # se conserva Fecha_Primera_Vigencia_Cert
    'vigencia_poliza',              # se conserva vigencia_certificado
    'Fecha_Recepcion',              # se conserva F_Notificacion
    'Fecha_Apertura',               # se conserva F_Notificacion
    'Periodo Reporte',              # se conserva Fecha de reporte
    'Año'                           # se conserva Fecha de reporte
]

Base_sin_duplicados = Base_sin_duplicados.drop(columns=columnas_redundantes)

print(f"Columnas eliminadas: {columnas_redundantes}")
print(f"Dimensiones actuales: {Base_sin_duplicados.shape}")

#Revisión variables categóricas
Revisamos las categorias de estas variables con el fin de:
- Encontrar categorías que significan lo mismo pero que estan escritas diferente
- Encontrar categorías que se puedan interpretar como NAN


In [0]:
columnas = [
    'Ramo_Desc', 'Nombre_plan',
    'Nombre_Canal_Comercial','SEXO_asegurado',
    'SUCURSAL', 'REGIONAL', 'AGENTE',
    'DESCAUSA', 'DIAGNOSTICO', 'Ind_Tipo_Atencion', 'Ind_Pago_Automatico',
    'estado', 'Cobertura', 'Tipo apertura', 'Fraude S /N'
]

for col in columnas:
    valores = Base_sin_duplicados[col].unique()
    print(f"\n{'='*60}")
    print(f"Columna: {col}  |  Valores únicos: {len(valores)}")
    print(f"{'='*60}")
    if len(valores) <= 100:
        print(valores)
    else:
        print(f"(demasiados para mostrar todos, primeros 20): {valores[:100]}")


Luego de examinar las variables con menor cardinalidad pude encontrar que en la variable **estado** cuneta con las opciones  **Anulado / ANULADO** , **Objetado / OBJETADO** y **Tramitado / TRAMITADO** las cuales con categorias que significan lo mismo pero estan escritas de manera diferente , lo mismo para la variable **Tipo apertura** que cuenta con las opciones **Oficina / oficina** , Con el fin de unificar dichas categorias , Paso a convertir todas las categorias de estas variables en mayúsculas 

In [0]:
Base_sin_duplicados['estado'] = Base_sin_duplicados['estado'].str.strip().str.upper()
Base_sin_duplicados['Tipo apertura'] = Base_sin_duplicados['Tipo apertura'].str.strip().str.upper()

print(Base_sin_duplicados['estado'].value_counts())
print(Base_sin_duplicados['Tipo apertura'].value_counts())


#Unificamos los valores nulos

Adicionalmente identificamos **diferentes valores** que pueden ser interpredados como nulos , esto ayuda bastante para poder medir com mayor exactitud que tan completas estan las variables.

In [0]:
Base_sin_duplicados.replace(["none","None", "null","NaN","nan","[]","{}",""," ","not_specified",None,"Sin Información","?"], np.nan, inplace=True)
pd.set_option('display.max_rows', None)

Porcetajes_na = Base_sin_duplicados.isnull().mean() * 100

df_porcentaje_nan = pd.DataFrame({'variable':Porcetajes_na.index, 'Porcentaje_na':Porcetajes_na.values})

df_porcentaje_nan.head(115)


Luego de unificar los **valores nulos** el panorama inicial no cambia mucho , ninguna variable supera el 1% de **datos nulos** detectados por el momento , por lo que afortunadamente hasta este momento todas las variables cuentan con la informacion suficiente , sin embargo  es necesario limpiar los pocos registros que contienen estos **datos nulos** , en total son solo **218 registros** por eliminar por el momento asi que procedo y quedamos con **12.229 registros** 

In [0]:
Base_sin_duplicados.isnull().any(axis=1).sum()

Base_sin_duplicados = Base_sin_duplicados.dropna()

print(f"Registros despues de eliminar: {len(Base_sin_duplicados)}")

#Revisión data leakage
Dado que el objetivo del modelo es categorizar posibles perfiles de fraude primero debemos analizar que variables estarían desponibles antes de que en la base se concluyera como caso de fraude o no , por lo que a continuacion procedo con el analisis y toma de accion pertinente según el caso

### Estado

Por definición, esta variable determina si un reclamo fue admitido o no. Como estamos trabajando con una base de datos en la que ya se determinó qué reclamaciones fueron fraude y cuáles no, lo más lógico es que esas reclamaciones hayan quedado como Objetadas o Anuladas —algo que no se podría saber sin antes haber determinado que se trataba de fraude—, y eso es justamente lo que muestran los datos: estas categorías están relacionadas en un 80% con casos de fraude. Por lo tanto, lo mejor es eliminar esta variable para evitar el riesgo de fuga de información.

In [0]:
es_objetado_anulado = Base_sin_duplicados['estado'].isin(['OBJETADO', 'ANULADO'])

tasa_fraude_objetado_anulado = Base_sin_duplicados[es_objetado_anulado]['Fraude S /N'].eq('Fraude').mean()
tasa_no_fraude_resto = Base_sin_duplicados[~es_objetado_anulado]['Fraude S /N'].eq('No es fraude').mean()

print(f"Cuando estado es Objetado o Anulado, tasa de fraude: {tasa_fraude_objetado_anulado:.1%}")
print(f"Cuando estado NO es Objetado ni Anulado, tasa de no-fraude: {tasa_no_fraude_resto:.1%}")

### Fecha_Primer_Cierre_Siniestro

Esta variable solo toma un valor real cuando el caso ya se cerro. Trabajando sobre una base donde ya se sabe que reclamaciones fueron fraude y cuales no, lo logico es que los casos ya resueltos sean los que mas se relacionan con fraude, algo que no se podria saber sin antes conocer el resultado del caso. Los datos lo confirman: los casos con fecha de cierre real tienen una tasa de fraude del 67.9%, contra 38.5% en los que aun estan abiertos, una diferencia de casi 30 puntos. Por eso se elimina, mismo riesgo de fuga de informacion que las anteriores.

In [0]:
fecha_cierre_parseada = pd.to_datetime(Base_sin_duplicados['Fecha_Primer_Cierre_Siniestro'], dayfirst=True, errors='coerce')
caso_cerrado = fecha_cierre_parseada.notnull() & (fecha_cierre_parseada.dt.year != 1900)

tasa_fraude_cerrado = Base_sin_duplicados[caso_cerrado]['Fraude S /N'].eq('Fraude').mean()
tasa_fraude_no_cerrado = Base_sin_duplicados[~caso_cerrado]['Fraude S /N'].eq('Fraude').mean()

print(f"Tasa de fraude cuando el caso tiene fecha de cierre real: {tasa_fraude_cerrado:.1%}")
print(f"Tasa de fraude cuando el caso NO tiene fecha de cierre real: {tasa_fraude_no_cerrado:.1%}")

### Sum(Valor_Pagos) y Sum(Valor_Reservas)

Estas variables tambien estan muy ligadas al resultado final del veredicto, dado que un siniestro calificado como fraude generalmente no se paga (hay varios valores de pago en 0). El valor final de la reserva tambien se ve afectado por esto: cuando el caso se cierra sin pago (por rechazo o anulacion), la reserva se libera igual, asi como cuando si hay pago tambien se libera. Es decir, la reserva liberada no distingue directamente si el caso fue pagado o rechazado, sino simplemente si el caso ya se cerro.

Mirando la relacion entre el no pago y los casos de fraude, encontramos una tasa del 65.6%, no tan alta como la de la variable estado, pero al separar la reserva liberada segun si hubo pago o no, el panorama se aclara mas: cuando se libera con pago, la tasa de fraude es de apenas 57.6% (casi neutral), pero cuando se libera sin pago (rechazo o anulacion) la tasa sube a 81.1%. Esto confirma que la reserva no mide fraude por si misma, sino si el caso ya fue resuelto, y los casos de fraude en esta base se resuelven con mas frecuencia (83.6%) que los legitimos (62.5%). Por esta razon, junto con el hecho de que ambas variables solo toman su valor definitivo despues de que el caso ya fue investigado, se consideran variables con riesgo de fuga de informacion (data leakage) y bajo este criterio seran eliminadas

In [0]:
pago_cero = Base_sin_duplicados['Sum(Valor_Pagos)'] == 0

tasa_fraude_pago_cero = Base_sin_duplicados[pago_cero]['Fraude S /N'].eq('Fraude').mean()
tasa_fraude_con_pago = Base_sin_duplicados[~pago_cero]['Fraude S /N'].eq('Fraude').mean()

print(f"Tasa de fraude cuando el pago es 0: {tasa_fraude_pago_cero:.1%}")
print(f"Tasa de fraude cuando SI hubo pago: {tasa_fraude_con_pago:.1%}")

In [0]:
reserva_cero = Base_sin_duplicados['Sum(Valor_Reservas)'] == 0
pago_cero = Base_sin_duplicados['Sum(Valor_Pagos)'] == 0

print("Tasa de fraude - reserva liberada con pago:", Base_sin_duplicados[reserva_cero & ~pago_cero]['Fraude S /N'].eq('Fraude').mean())
print("Tasa de fraude - reserva liberada sin pago:", Base_sin_duplicados[reserva_cero & pago_cero]['Fraude S /N'].eq('Fraude').mean())
print("Tasa de fraude - reserva no liberada:", Base_sin_duplicados[~reserva_cero]['Fraude S /N'].eq('Fraude').mean())

### Ind_Pago_Automatico

Al igual que las demás variables, considero que esta está muy ligada al evento del pago; es decir, sin que se haya realizado el pago no es posible determinar si este fue manual o automático. Por eso considero que esta variable se construye después de la clasificación de fraude. Otra evidencia de esto es su relación con la variable **estado** (que ya identificamos como fuerte data leakage): existe una relación de casi el 100% entre los pagos no automáticos y los siniestros en estado Anulado u Objetado. Por lo tanto, esta variable también corre el riesgo de ser data leakage y debe ser eliminada

In [0]:
categorias_resolucion = ['ANULADO', 'OBJETADO']
casos_resolucion = Base_sin_duplicados[Base_sin_duplicados['estado'].isin(categorias_resolucion)]

print(casos_resolucion['Ind_Pago_Automatico'].value_counts(normalize=True))

Procedemos con la eliminacion de las variables con riesgo data leakage y nos quedamos con 25 de las 40 columnas con que iniciamos el análisis

In [0]:
columnas_leakage = ['estado', 'Sum(Valor_Pagos)', 'Sum(Valor_Reservas)', 'Ind_Pago_Automatico','Fecha_Primer_Cierre_Siniestro']

print(f"Registros antes: {Base_sin_duplicados.shape}")

Base_sin_duplicados = Base_sin_duplicados.drop(columns=columnas_leakage)

print(f"Columnas eliminadas: {columnas_leakage}")
print(f"Dimensiones actuales: {Base_sin_duplicados.shape}")

#Estadística Descriptiva
Con la función **ProfileReport** podemos implementar un **análisis descriptivo** profundo tanto para las variables numéricas como categóricas , aprovecharemos esta visual para conocer la **distrubución** de nuestra variables, **calidad** y **atípicos** , esto nos va a permitir refinar aun mas la limpieza de registros y variables que empezamos en las secciones anteriores

In [0]:
descripcion_estadistica = ProfileReport(Base_sin_duplicados, title="Profiling Report")
descripcion_estadistica

### Hallazgos de la Estadística Descriptiva

1. Tras haber eliminado gran cantidad de columnas, nos encontramos con 243 registros duplicados, los cuales serán eliminados.

2. Se elimina la variable **Amparo_Desc**, ya que guarda una correlación alta con las variables **Cobertura** y **Ramo**. Con **Cobertura** basta, y además presenta menor cardinalidad.

3. Las variables **edad_actual_asegurado** y **edad_ingreso_asegurado** guardan una similitud del 93%. Por esta razón me quedo únicamente con **edad_actual_asegurado**. Pensaba usar ambas variables para calcular la antigüedad, pero dado que son tan similares y ya cuento con **vigencia_certificado**, opto por eliminar una de las dos.

4. Existe una correlación alta entre **Ind_Tipo_Atencion** y **Tipo apertura**; al parecer explican lo mismo, pero de forma diferente. Me quedo con **Tipo apertura** porque resulta mucho más explicativa.

5. Mirando la distribución de la variable **edad_actual_asegurado** se detecta un valor atípico que corresponde a la categoría -1 solo son 46 registros con esta condicion así que los eliminaremos. 



Las variables **DIAGNOSTICO** , **AGENTE** , **SUCURSAL** y **Nombre_Canal_Comercial** tienen una alta cardinalidad por lo que la libreria las excluyo del analisis de correlacion , por ello decidi hacer aparte una matriz de correlacion para examinalas frente a las variables que por definición guardaria una correlación alta, vamos a validar esa hipotesis

In [0]:
import phik

columnas_correlacion = ['DESCAUSA', 'DIAGNOSTICO', 'AGENTE', 'SUCURSAL', 
                         'REGIONAL', 'Nombre_Canal_Comercial', 'Fraude S /N']

matriz_alta_cardinalidad = Base_sin_duplicados[columnas_correlacion].phik_matrix(interval_cols=[])

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(matriz_alta_cardinalidad, annot=True, fmt='.3f', cmap='coolwarm', 
            vmin=0, vmax=1, square=True, linewidths=0.5)
plt.title("Correlacion (phik) - Variables de alta cardinalidad vs Fraude S /N")
plt.tight_layout()
plt.show()

### Hallazgos matriz de correlación

**DESCAUSA vs DIAGNOSTICO:** Ambas variables tienen una correlacion de 0.9997 entre si, practicamente identicas. Sin embargo, DIAGNOSTICO tiene una relacion mucho mas fuerte con la variable objetivo (0.837) que DESCAUSA (0.572). Aunque DIAGNOSTICO tiene mucha mas cardinalidad (1478 categorias contra 28 de DESCAUSA), se conserva DIAGNOSTICO y se elimina DESCAUSA, porque se pierde muy poco en simplicidad y se gana bastante poder predictivo.



**AGENTE:** Se elimina por dos razones. Primero, tiene una cardinalidad muy alta (1079 categorias), lo que implica mas riesgo de sobreajuste y mayor costo computacional en los proximos modelos y tiene una alta correlacion con SUCURSAL, canal comercial y regional asi que no aporta informacion nueva que estas otras no cubran.


**SUCURSAL, REGIONAL y Nombre_Canal_Comercial:** Las tres variables estan altamente correlacionadas entre si (SUCURSAL-REGIONAL: 1.000, SUCURSAL-Nombre_Canal_Comercial: 0.999). Comparando cada una contra la variable objetivo:

- SUCURSAL: 0.672
- Nombre_Canal_Comercial: 0.476
- REGIONAL: 0.284

Se conserva unicamente SUCURSAL, ya que es la que mayor relacion tiene con el fraude. Eliminar REGIONAL y Nombre_Canal_Comercial no representa perdida real de informacion, porque esta ya esta contenida en SUCURSAL, que ademas ofrece un nivel de detalle mas especifico.

Tras los hallazgos encontrados, procedemos a eliminar las variables ya mencionadas., tas esto nos quedamos con 17 variables de las 40 originales

In [0]:
columnas_correlacionadas = [
    'Amparo_Desc',
    'edad_ingreso_asegurado',
    'Ind_Tipo_Atencion',
    'DESCAUSA',
    'AGENTE',
    'REGIONAL',
    'Nombre_Canal_Comercial'
]

print(f"Dimensiones antes: {Base_sin_duplicados.shape}")

Base_sin_duplicados = Base_sin_duplicados.drop(columns=columnas_correlacionadas)

print(f"Columnas eliminadas: {columnas_correlacionadas}")
print(f"Dimensiones actuales: {Base_sin_duplicados.shape}")

Eliminamos registros con **edad_actual_asegurado** en -1

In [0]:
print(f"Registros antes: {len(Base_sin_duplicados)}")
print(f"Registros con edad_actual_asegurado = -1: {(Base_sin_duplicados['edad_actual_asegurado']==-1).sum()}")

Base_sin_duplicados = Base_sin_duplicados[Base_sin_duplicados['edad_actual_asegurado'] != -1]

print(f"Registros despues: {len(Base_sin_duplicados)}")

Adicionalmente tras haber modificado la base eliminamos registros duplicados , nos quedamos con 11891 registros

In [0]:
print(f"Registros antes de eliminar duplicados: {len(Base_sin_duplicados)}")
print(f"Duplicados encontrados: {Base_sin_duplicados.duplicated().sum()}")

Base_sin_duplicados = Base_sin_duplicados.drop_duplicates()

print(f"Registros despues de eliminar duplicados: {len(Base_sin_duplicados)}")

In [0]:
Base_sin_duplicados.info()

#Creacion de nuevas variables 
Usando como insumo variables que no generan valor por si solas creamos nuevas variables que tendrán un gran potencial en la modelación , las ideas son las siguientes:

- **dias_desde_ultima_reclamacion:** Aprovechando que aún no eliminamos la *identificacion del cliente* y en conjunto con la variable *FSINIESTRO*, podemos calcular registro a registro cuantos dias han pasado desde la ultima reclamacion (Lo interesante de esta varieable es que podemos apreciar si hay relacion entre los siniestros reiterados y los casos de fraude).

- **num_reclamaciones_previas:** Usamos las mismas variables que en el caso anterior anterior para calcular cuantas reclamaciones ha tenido la persona antes de la actual.

- **dias_vigencia_hasta_siniestro:** A partir de las variables *FSINIESTRO* y *Fecha_Primera_Vigencia_Cert* Vamos a calcular cuantos dias han pasado entre el inicio de la poliza y la ocurrencia del siniestro (esta variable me parece interesante sobretodo para identificar a las personas que solo compran el seguro para inmediatamente siniestrarse)

- **dias_siniestro_a_notificacion:** Con la variable *FSINIESTRO* y *F_Notificacion* calculamos cuantos dias pasaron entre la ocurrencia y el aviso del siniestro.

- **Rango_edad:** categorizamos la variable *edad_actual_asegurado* por facilidad en la futura interpretacion de los resultados

- **diagnostico_agrupado:** ya que la Variable *DIAGNOSTICO* posee una alta cardinalidad y muchas de sus categorías aparecen muy pocas veces , dichas categorías quedaran en una bolsa de "otros"

- **reserva_inicial_relativa:** dado que *Sum(Valor_Reservas_Inicial)* es un indicador numerico no tan sencilo de tratar , con tanto valores altos como 0 , decidi construir una variable que va a comparar el valor promedio de dicha reserva por *DIAGNOSTICO* contra el valor de la reserva inicial en ese momento.

###dias_desde_ultima_reclamacion
**Nota:** cuando es la primera reclamacion lo deje con -1 para que no dañe el formato

In [0]:
# Ordenar por asegurado y fecha de siniestro para poder calcular el historial correctamente
Base_sin_duplicados = Base_sin_duplicados.sort_values(['IDENTIFICACION_asegurado', 'FSINIESTRO'])

# Fecha del siniestro anterior de ese mismo asegurado
fecha_siniestro_anterior = Base_sin_duplicados.groupby('IDENTIFICACION_asegurado')['FSINIESTRO'].shift(1)

# Diferencia en dias
Base_sin_duplicados['dias_desde_ultima_reclamacion'] = (
    Base_sin_duplicados['FSINIESTRO'] - fecha_siniestro_anterior
).dt.days

# Los nulos (primera reclamacion de cada asegurado) se marcan como -1, y todo queda en entero
Base_sin_duplicados['dias_desde_ultima_reclamacion'] = Base_sin_duplicados['dias_desde_ultima_reclamacion'].fillna(-1).astype(int)

# Verificacion rapida: mira un asegurado con varias reclamaciones
print(Base_sin_duplicados[Base_sin_duplicados.duplicated('IDENTIFICACION_asegurado', keep=False)]
      [['IDENTIFICACION_asegurado','FSINIESTRO','dias_desde_ultima_reclamacion']]
      .sort_values(['IDENTIFICACION_asegurado','FSINIESTRO'])
      .head(10))

###num_reclamaciones_previas

In [0]:
# Numero de reclamaciones previas del mismo asegurado (sin contar la actual)
Base_sin_duplicados['num_reclamaciones_previas'] = Base_sin_duplicados.groupby('IDENTIFICACION_asegurado').cumcount()

# Verificacion rapida: mismo asegurado con varias reclamaciones
print(Base_sin_duplicados[Base_sin_duplicados.duplicated('IDENTIFICACION_asegurado', keep=False)]
      [['IDENTIFICACION_asegurado','FSINIESTRO','num_reclamaciones_previas']]
      .sort_values(['IDENTIFICACION_asegurado','FSINIESTRO'])
      .head(10))

###dias_vigencia_hasta_siniestro
hay inconsistencias en las fechas por que hay siniestros que ocurrieron antes de el inicio de la vigencia son pocos casos asi que se imputan por la moda

In [0]:
# Calcular la variable
Base_sin_duplicados['dias_vigencia_hasta_siniestro'] = (
    Base_sin_duplicados['FSINIESTRO'] - Base_sin_duplicados['Fecha_Primera_Vigencia_Cert']
).dt.days

# Calcular la mediana solo con los valores validos (no negativos)
mediana_dias_vigencia = Base_sin_duplicados.loc[
    Base_sin_duplicados['dias_vigencia_hasta_siniestro'] >= 0, 
    'dias_vigencia_hasta_siniestro'
].median()

print(f"Mediana calculada (sin negativos): {mediana_dias_vigencia}")
print(f"Registros negativos a reemplazar: {(Base_sin_duplicados['dias_vigencia_hasta_siniestro']<0).sum()}")

# Reemplazar los valores negativos por la mediana
Base_sin_duplicados.loc[
    Base_sin_duplicados['dias_vigencia_hasta_siniestro'] < 0, 
    'dias_vigencia_hasta_siniestro'
] = mediana_dias_vigencia

# Verificacion rapida
print(Base_sin_duplicados[['IDENTIFICACION_asegurado','Fecha_Primera_Vigencia_Cert','FSINIESTRO','dias_vigencia_hasta_siniestro']].head(10))

###dias_siniestro_a_notificacion

In [0]:
# Calcular la variable
Base_sin_duplicados['dias_siniestro_a_notificacion'] = (
    Base_sin_duplicados['F_Notificacion'] - Base_sin_duplicados['FSINIESTRO']
).dt.days

# Calcular la mediana solo con los valores validos (no negativos)
mediana_dias_notificacion = Base_sin_duplicados.loc[
    Base_sin_duplicados['dias_siniestro_a_notificacion'] >= 0,
    'dias_siniestro_a_notificacion'
].median()

print(f"Mediana calculada (sin negativos): {mediana_dias_notificacion}")
print(f"Registros negativos a reemplazar: {(Base_sin_duplicados['dias_siniestro_a_notificacion']<0).sum()}")

# Reemplazar los valores negativos por la mediana
Base_sin_duplicados.loc[
    Base_sin_duplicados['dias_siniestro_a_notificacion'] < 0,
    'dias_siniestro_a_notificacion'
] = mediana_dias_notificacion

# Verificacion rapida
print(Base_sin_duplicados[['IDENTIFICACION_asegurado','FSINIESTRO','F_Notificacion','dias_siniestro_a_notificacion']].head(10))

###Rango_edad

In [0]:
Base_sin_duplicados['Rango_edad'] = pd.cut(
    Base_sin_duplicados['edad_actual_asegurado'],
    bins=[0, 18, 28, 40, 60, 100],
    labels=['0-18', '19-28', '29-40', '41-60', '>60'],
    include_lowest=True
)

# Verificacion rapida
print(Base_sin_duplicados['Rango_edad'].value_counts().sort_index())
print()
print(Base_sin_duplicados[['IDENTIFICACION_asegurado','edad_actual_asegurado','Rango_edad']].head(10))

###diagnostico_agrupado
Nos quedamos con el top 30 de los diagnósticos y el resto se queda en la bolsa de Otros , con estas del top 30 seguimos manteniendo una correlacion alta con la variable objetivo


In [0]:
top30 = Base_sin_duplicados['DIAGNOSTICO'].value_counts().head(30).index
en_top30 = Base_sin_duplicados['DIAGNOSTICO'].isin(top30)

print("Fraude en Top 30:", Base_sin_duplicados[en_top30]['Fraude S /N'].eq('Fraude').mean())
print("Fraude en Otros:", Base_sin_duplicados[~en_top30]['Fraude S /N'].eq('Fraude').mean())

In [0]:
top_30_diagnosticos = Base_sin_duplicados['DIAGNOSTICO'].value_counts().head(30).index

Base_sin_duplicados['diagnostico_agrupado'] = Base_sin_duplicados['DIAGNOSTICO'].where(
    Base_sin_duplicados['DIAGNOSTICO'].isin(top_30_diagnosticos), 
    'Otros'
)

# Verificacion
print(Base_sin_duplicados['diagnostico_agrupado'].value_counts())

###reserva_inicial_relativa

In [0]:
# Promedio de reserva inicial por grupo de diagnostico (agrupado)
promedio_reserva_por_diagnostico = Base_sin_duplicados.groupby('diagnostico_agrupado')['Sum(Valor_Reservas_Inicial)'].transform('mean')

# Variable: que tanto se aleja la reserva de este caso respecto al promedio de su grupo
Base_sin_duplicados['reserva_inicial_relativa'] = (
    Base_sin_duplicados['Sum(Valor_Reservas_Inicial)'] / promedio_reserva_por_diagnostico
)

# Verificacion rapida
print(Base_sin_duplicados[['diagnostico_agrupado','Sum(Valor_Reservas_Inicial)','reserva_inicial_relativa']].head(10))

## Eliminación de variables sobrantes

Dado que muchas de las variables que decidimos conservar al principio, por su potencial para construir otras variables de valor, ya cumplieron su función, procedemos a eliminarlas y quedarnos únicamente con las nuevas. (luego de haber creado las neuvas variables y eliminado las que nos sobran quedamos con 15 variables)

In [0]:
columnas_ya_transformadas = [
    'IDENTIFICACION_asegurado',
    'FSINIESTRO',
    'F_Notificacion',
    'Fecha_Primera_Vigencia_Cert',
    'Sum(Valor_Reservas_Inicial)',
    'DIAGNOSTICO',
    'edad_actual_asegurado',
    'FEXPEDICION',
    'Fecha de reporte'
]

print(f"Dimensiones antes: {Base_sin_duplicados.shape}")

Base_sin_duplicados = Base_sin_duplicados.drop(columns=columnas_ya_transformadas)

print(f"Columnas eliminadas: {columnas_ya_transformadas}")
print(f"Dimensiones actuales: {Base_sin_duplicados.shape}")

In [0]:
Base_sin_duplicados.info()

#Segundo analisis de correlación
Dado que transformamos por completo nuestra base de datos con la creacion de nuevas variables , es necesario volver a analizar la correlacion entre ellas para determinar si colinealidad en alguna de ellas

In [0]:
import phik
import matplotlib.pyplot as plt
import seaborn as sns

matriz_final = Base_sin_duplicados.phik_matrix(interval_cols=[
    'vigencia_certificado', 'Sum(Valor_Reservas_Inicial)', 
    'dias_desde_ultima_reclamacion', 'num_reclamaciones_previas',
    'dias_vigencia_hasta_siniestro', 'dias_siniestro_a_notificacion',
    'reserva_inicial_relativa'
])

plt.figure(figsize=(14, 12))
sns.heatmap(matriz_final, annot=True, fmt='.2f', cmap='coolwarm', 
            vmin=0, vmax=1, square=True, linewidths=0.5)
plt.title("Matriz de correlacion final (phik)")
plt.tight_layout()
plt.show()

## Interpretacion de la matriz de correlacion final y limpieza

### Variables que se deberian eliminar por alta correlacion entre si

- Ramo_Desc y Nombre_plan tienen correlacion perfecta (1.00), son la misma informacion.
- Nombre_plan y SUCURSAL tambien estan practicamente pegadas (0.99).
- Entre las tres (Ramo_Desc, Nombre_plan, SUCURSAL) me quedo con SUCURSAL y elimino las otras dos, porque es la que mas correlacion tiene con Fraude S /N (0.67), por encima de Nombre_plan (0.62) y Ramo_Desc (0.53).
- vigencia_certificado y dias_vigencia_hasta_siniestro tambien estan casi duplicadas (0.97). Me quedo con dias_vigencia_hasta_siniestro y elimino vigencia_certificado, porque tiene levemente mayor correlacion con el objetivo (0.15 vs 0.11).

### Variables con mayor correlacion frente al objetivo

De mayor a menor: SUCURSAL (0.67), Nombre_plan (0.62), Tipo apertura (0.60), diagnostico_agrupado (0.56), Ramo_Desc (0.53) y dias_siniestro_a_notificacion (0.45). Estas seis son las que mas peso deberian tener en el modelo.

### Como se comportan las variables nuevas

- dias_siniestro_a_notificacion es la variable nueva con mejor desempeño, 0.45 con el objetivo. Confirma la hipotesis de que el tiempo de aviso del siniestro si es un patron util para detectar fraude.
- diagnostico_agrupado tambien queda bien, 0.56 con el objetivo, lo que confirma que agrupar en Top 30 + Otros no sacrifico demasiada señal.
- num_reclamaciones_previas tiene una correlacion moderada (0.27), la reincidencia del asegurado si aporta algo, aunque no tanto como esperaba.
- dias_vigencia_hasta_siniestro y Rango_edad quedan bastante debiles frente al objetivo (0.15 y 0.07).
- dias_desde_ultima_reclamacion y reserva_inicial_relativa son las mas debiles de todas las variables nuevas, con apenas 0.02 de correlacion con el objetivo, y tambien correlacionan poco con el resto de variables. vamos a eliminarlas de una vez para evitar rerpocesos.

Realizamos la ultima limpieza a nivel de variables y empezamos con la preparacion de datos para la modelación , quedamos con 10 variables predictoras y 1 objetivo

In [0]:
columnas_a_eliminar_final = [
    'Ramo_Desc',
    'Nombre_plan',
    'vigencia_certificado',
    'dias_desde_ultima_reclamacion',
    'reserva_inicial_relativa'
]

print(f"Dimensiones antes: {Base_sin_duplicados.shape}")

Base_sin_duplicados = Base_sin_duplicados.drop(columns=columnas_a_eliminar_final)

print(f"Columnas eliminadas: {columnas_a_eliminar_final}")
print(f"Dimensiones actuales: {Base_sin_duplicados.shape}")
print()
print(Base_sin_duplicados.columns.tolist())

In [0]:
Base_sin_duplicados.info()

#Modelación

## 1.DecisionTreeClassifier
El primero en implementar es un arbol de decisión ya que es un modelo rápido , facil de configurar e interpretar , se ajusta muy bien a los problemas de clasificación y permite visualizar las diferentes combinaciones

**Desventajas:** Se sobreajusta fácil, memoriza en vez de aprender un patrón general. Y es inestable: con cambiar un poco los datos, las reglas del árbol pueden cambiar bastante.

####**Preparación de los datos**

In [0]:
#creamos una copia de la base original 
Base_arbol = Base_sin_duplicados.copy()

#vamos a codificar la variable objetivo ya que sklearn no acepta variables categoricas
Base_arbol['Fraude S /N'] = Base_arbol['Fraude S /N'].map({'No es fraude': 0, 'Fraude': 1})

#de paso codificamos las categorias que no tienen ningun orden natural

from sklearn.preprocessing import LabelEncoder

columnas_categoricas = ['SEXO_asegurado', 'SUCURSAL', 'Cobertura', 'Tipo apertura', 'diagnostico_agrupado']

for col in columnas_categoricas:
    le = LabelEncoder()
    Base_arbol[col] = le.fit_transform(Base_arbol[col])
#asignamos un orden por numero a la unica variable categorica ordinal que tenemos

orden_edad = {'0-18': 0, '19-28': 1, '29-40': 2, '41-60': 3, '>60': 4}
Base_arbol['Rango_edad'] = Base_arbol['Rango_edad'].map(orden_edad)
Base_arbol.head(5)

####**Creación del conjunto de prueba y el conjunto de entrenamiento**
Hacemos una proporcion 80-20 entre la base de entrenamiento y la de prueba , en ambas bases se mantiene un equilibrio de la variable objetivo

In [0]:
from sklearn.model_selection import train_test_split

X = Base_arbol.drop(columns=['Fraude S /N'])
y = Base_arbol['Fraude S /N']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Proporcion de fraude en train: {y_train.mean():.2%}")
print(f"Proporcion de fraude en test: {y_test.mean():.2%}")

####**Entrenamiento del modelo**
Entrenamos nuestro arbol de decisión con el plus de que estamo haciendo hiperparametrización , esto quiere decir que estamos usando los parametros que nos llevan a la mejor calificación del modelo , adicionalmente guardamos la semilla para que aparezcan los mismos resultados siempre que se ejecute.

In [0]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# Crear el modelo de arbol de decision
clf = DecisionTreeClassifier(random_state=42)

# Definir los hiperparametros que se desean ajustar
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [2, 5, 10]
}

# Buscar la mejor combinacion
grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=5)
grid_search.fit(X_train, y_train)

# Obtener los mejores hiperparametros
best_params = grid_search.best_params_
print("Mejores hiperparametros:", best_params)

# Entrenar el modelo con los mejores hiperparametros
best_clf = DecisionTreeClassifier(random_state=42, **best_params)
best_clf.fit(X_train, y_train)

####**Evaluación del modelo**
El árbol de decisión logra un desempeño sólido como primer modelo: identifica correctamente 3 de cada 4 casos de fraude, y cuando marca una alarma, acierta el 86% de las veces. Aún deja pasar algunos fraudes sin detectar, pero es un punto de partida confiable para comparar contra modelos más robustos.

In [0]:
from sklearn import metrics

# Evaluar el modelo
y_pred_best = best_clf.predict(X_test)

acc = metrics.accuracy_score(y_test, y_pred_best)
print(f'Exactitud: {acc}')

matconf = metrics.confusion_matrix(y_test, y_pred_best, labels=[1,0])  # filas:real, col:prediccion
print(f'Matriz de Confusion (filas:real, col:prediccion): \n{matconf}')

p = metrics.precision_score(y_test, y_pred_best, labels=[1,0], average=None)
print(f'Precision: {p}')

r = metrics.recall_score(y_test, y_pred_best, labels=[1,0], average=None)
print(f'Cobertura (recall): {r}')

f1 = metrics.f1_score(y_test, y_pred_best, labels=[1,0], average=None)
print(f'F1: {f1}')

fpr, tpr, thresholds = metrics.roc_curve(y_test, y_pred_best, pos_label=1)  # pos_label indica la clase positiva
plt.plot(fpr, tpr)

auc = metrics.auc(fpr, tpr)
print(f'Area bajo la curva ROC: {auc}')

### Interpretación del árbol de decisión

**Variable raíz (la más importante de todo el modelo):**

**`dias_siniestro_a_notificacion`** es la primera pregunta que hace el árbol para dividir los casos. Notificaciones que tardan más de 229 días empujan fuertemente hacia fraude, las más rápidas empujan hacia legítimo. Coincide con la regresión, donde esta misma variable resultó ser la de mayor odds ratio (11.7x) de todo el modelo.

**Segunda variable clave, presente en ambas ramas del árbol:**

**`Tipo apertura`** aparece justo debajo de la variable raíz en las dos ramas principales, confirmando que es de las más influyentes. Coincide con la regresión, donde esta variable también mostró los odds ratio más marcados entre las categóricas.

**Perfil de altísimo riesgo (98% fraude, 2552 de 2591 casos):**

Notificación tardía (más de 229 días) + apertura por **Oficina** o **Reclamaciones Digital** + cobertura de tipo **Ap Complementario**, **Especiales** o **Invalidez** (cualquiera excepto Renta o Vida). Es el segmento más puro de todo el árbol.

**Perfil de riesgo moderado-bajo (66% no es fraude, 2468 de 3763 casos):**

Notificación rápida + apertura por **Asesor**, **Digitador Externo**, **Digitador Sucursal**, **Línea** u **Oficina** (cualquiera excepto Reclamaciones Digital) + siniestro ocurrido dentro de los primeros ~240 días de vigencia.

**Canal digital como señal de riesgo propia (60% fraude, 1742 de 2903 casos):**

Notificación rápida + apertura específicamente por **Reclamaciones Digital**, tiende más hacia fraude que hacia legítimo sin importar qué tan rápido se notificó. El canal digital parece sospechoso por sí mismo.

**Conclusión general:**

**`Tipo apertura`** es determinante en casi todas las ramas: **Reclamaciones Digital** y **Oficina** concentran el riesgo, mientras que **Asesor** y **Digitador** (canales más tradicionales) se asocian más a casos legítimos, salvo cuando se combinan con notificación tardía. **`Cobertura`** también discrimina fuerte: **Vida** y **Renta** parecen más seguras, el resto concentra el riesgo cuando se combina con los otros factores. Coincide con lo que confirmó despues la regresión: mismas variables, misma dirección del efecto, solo que la regresión da el número exacto de cuantas veces más, en vez de solo más o menos.

In [0]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(24, 12))
plot_tree(
    best_clf,
    feature_names=X_train.columns,
    class_names=['No es fraude', 'Fraude'],
    filled=True,
    rounded=True,
    fontsize=9,
    max_depth=3
)
plt.show()

####**Traductor de variables categóricas**
como se explica al principio  sklearn trabaja unicamente con variables categóricas por lo que es necesario traducir las variables a su categoría original para poder dar una interpretacion completa. 

In [0]:
import pandas as pd

for col in ['SEXO_asegurado', 'SUCURSAL', 'Cobertura', 'Tipo apertura', 'diagnostico_agrupado']:
    categorias_ordenadas = sorted(Base_sin_duplicados[col].unique())
    print(f"\n{col}:")
    for i, cat in enumerate(categorias_ordenadas):
        print(f"  {i} -> {cat}")

##2.LogisticRegression
El segundo modelo a implementar es una regresión logística, la elijo porque a diferencia de los modelos de árboles ofrece coeficientes interpretables como probabilidad directamente, lo que permite entender de forma clara el peso e influencia de cada variable sobre el riesgo de fraude y construir perfiles de negocio fáciles de comunicar.

**Desventajas:** Asume que las relaciones son lineales, así que si el efecto real de una variable no es una línea recta, se le escapa. Y con variables categóricas de muchas categorías (como sucursal) los coeficientes se vuelven poco confiables cuando hay pocos casos por categoría.

####**Preparación de los datos** 
Algo particular de la regresion es que para las variables categoricas transformadas en numero, reconoce siempre el orden del numero de forma estricta, por lo que puede causar interpretaciones erroneas. Por eso cada categoria de las variables categoricas se transforma en variables dummy, esto aumenta la dimension de la base pero permite que puedan ser usadas para la regresion.

Con sucursal paso algo particular, es nuestra variable con mayor cardinalidad con mas de 200 categorias. Si la transformamos directamente en dummies la dimension de la base crece considerablemente, por lo que primero la recategorizamos quedandonos con el top 20 de categorias mas comunes y el resto se deja en la bolsa de otros.

In [0]:
#creamos una copia de la base original
Base_regresion = Base_sin_duplicados.copy()

#vamos a codificar la variable objetivo ya que sklearn no acepta variables categoricas
Base_regresion['Fraude S /N'] = Base_regresion['Fraude S /N'].map({'No es fraude': 0, 'Fraude': 1})

#agrupamos SUCURSAL en Top 20 + Otros, dado que su alta cardinalidad (211 categorias)
#generaria demasiadas columnas con one-hot y volveria los coeficientes ilegibles
top_20_sucursales = Base_regresion['SUCURSAL'].value_counts().head(20).index
Base_regresion['sucursal_agrupada'] = Base_regresion['SUCURSAL'].where(
    Base_regresion['SUCURSAL'].isin(top_20_sucursales),
    'Otros'
)
Base_regresion = Base_regresion.drop(columns=['SUCURSAL'])

#a diferencia del arbol, la regresion logistica si necesita one-hot encoding
#para las categorias sin orden natural, ya que LabelEncoder inventaria un orden falso
#que la regresion si interpretaria como una relacion numerica

columnas_categoricas = ['SEXO_asegurado', 'sucursal_agrupada', 'Cobertura', 'Tipo apertura', 'diagnostico_agrupado']

Base_regresion = pd.get_dummies(Base_regresion, columns=columnas_categoricas, drop_first=True)

#asignamos un orden por numero a la unica variable categorica ordinal que tenemos
orden_edad = {'0-18': 0, '19-28': 1, '29-40': 2, '41-60': 3, '>60': 4}
Base_regresion['Rango_edad'] = Base_regresion['Rango_edad'].map(orden_edad)

Base_regresion.head(5)

####**Creación del conjunto de prueba y el conjunto de entrenamiento**
Hacemos una proporcion 80-20 entre la base de entrenamiento y la de prueba , en ambas bases se mantiene un equilibrio de la variable objetivo , se puede notar como el numero de columnas es mucho mas grande que en el arbol de decisión

In [0]:
from sklearn.model_selection import train_test_split

X = Base_regresion.drop(columns=['Fraude S /N'])
y = Base_regresion['Fraude S /N']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Proporcion de fraude en train: {y_train.mean():.2%}")
print(f"Proporcion de fraude en test: {y_test.mean():.2%}")

####**Escalado de variables**
Debemos normalizar la variables numericas que contienen valores muy grandes o desequilibrados a comparación de las demas esto con el fin de poner todas las variables numéricas en una escala comparable , para que una variable no domine a otra solo por tener numeros mas grandes en magnitud.


In [0]:
from sklearn.preprocessing import StandardScaler

columnas_numericas = ['num_reclamaciones_previas', 'dias_vigencia_hasta_siniestro', 'dias_siniestro_a_notificacion']

scaler = StandardScaler()

# Ajustamos el scaler SOLO con train, para no filtrar informacion del test
X_train[columnas_numericas] = scaler.fit_transform(X_train[columnas_numericas])
X_test[columnas_numericas] = scaler.transform(X_test[columnas_numericas])

print(X_train[columnas_numericas].describe())

####**Entrenamiento del modelo**
Entrenamos nuestra regresión con el plus de que estamo haciendo hiperparametrización , esto quiere decir que estamos usando los parametros que nos llevan a la mejor calificación del modelo , adicionalmente guardamos la semilla para que aparezcan los mismos resultados siempre que se ejecute.

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Crear el modelo de regresion logistica
lr = LogisticRegression(random_state=42, max_iter=1000)

# Definir los hiperparametros que se desean ajustar
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']  # liblinear soporta tanto l1 como l2
}

# Buscar la mejor combinacion
grid_search = GridSearchCV(estimator=lr, param_grid=param_grid, cv=5, scoring='f1')
grid_search.fit(X_train, y_train)

# Obtener los mejores hiperparametros
best_params = grid_search.best_params_
print("Mejores hiperparametros:", best_params)

# Entrenar el modelo con los mejores hiperparametros
best_lr = LogisticRegression(random_state=42, max_iter=1000, **best_params)
best_lr.fit(X_train, y_train)

####**Evaluación del modelo**
La regresión logística logra un desempeño competitivo frente al árbol: detecta 4 de cada 5 casos de fraude (recall del 82%), superando al árbol en esta métrica. Sin embargo, cuando marca una alarma acierta el 72% de las veces, por debajo del 86% del árbol, lo que significa más falsas alarmas para el equipo de investigación. En balance general (F1), el árbol sigue liderando levemente, aunque la regresión ofrece una capacidad de discriminación algo mejor (AUC de 0.81 frente a 0.79). Es un modelo más agresivo detectando fraude, a costa de mayor ruido en las alertas.

In [0]:
from sklearn import metrics
import matplotlib.pyplot as plt

# Evaluar el modelo
y_pred_best = best_lr.predict(X_test)

acc = metrics.accuracy_score(y_test, y_pred_best)
print(f'Exactitud: {acc}')

matconf = metrics.confusion_matrix(y_test, y_pred_best, labels=[1,0])  # filas:real, col:prediccion
print(f'Matriz de Confusion (filas:real, col:prediccion): \n{matconf}')

p = metrics.precision_score(y_test, y_pred_best, labels=[1,0], average=None)
print(f'Precision: {p}')

r = metrics.recall_score(y_test, y_pred_best, labels=[1,0], average=None)
print(f'Cobertura (recall): {r}')

f1 = metrics.f1_score(y_test, y_pred_best, labels=[1,0], average=None)
print(f'F1: {f1}')

fpr, tpr, thresholds = metrics.roc_curve(y_test, y_pred_best, pos_label=1)
plt.plot(fpr, tpr)

auc = metrics.auc(fpr, tpr)
print(f'Area bajo la curva ROC: {auc}')


### Interpretación de la regresión logística

**Variables numéricas (relación continua, no por categoría):**

* A mayor número de días entre el siniestro y su notificación, mayor la probabilidad de fraude. Por cada unidad que aumenta esta variable (ya estandarizada), el riesgo se multiplica por 11.7 veces — es la relación más fuerte de todo el modelo. Notificar tarde es la señal de alarma más potente que encontró la regresión, y coincide con el árbol de decisión, donde esta misma variable fue justo el primer corte (la más importante de todas).
* A mayor número de reclamaciones previas del asegurado, mayor la probabilidad de fraude. Cada aumento en reincidencia casi duplica el riesgo (1.94x). Confirma que los clientes reincidentes son más propensos a estar involucrados en fraude.
* A mayor tiempo entre el inicio de la vigencia de la póliza y la ocurrencia del siniestro, MENOR la probabilidad de fraude (0.66x, relación inversa). Dicho al revés, que es como suele presentarse este hallazgo en el negocio: entre más reciente sea la póliza al momento del siniestro, mayor es la probabilidad de fraude — confirma la sospecha clásica de "compró el seguro para reclamarlo casi enseguida".

**Variables categóricas (respecto a su categoría de referencia):**

* Si la apertura del caso fue por Reclamaciones Digital, el riesgo de fraude es 3.19 veces mayor que si fue por Asesor. Coincide con el árbol, donde este mismo canal aparecía como uno de los de mayor riesgo.
* Si la apertura fue por Oficina, el riesgo es 2.85 veces mayor que por Asesor. También coincide con el árbol.
* Si la apertura fue por Digitador Externo, el riesgo es la mitad (0.50x) comparado con Asesor — es de los canales más seguros, consistente con lo que ya mostraba el árbol para los canales tradicionales.
* Si la cobertura reclamada es Vida, el riesgo es menos de la mitad (0.44x) comparado con Ap Complementario. Coincide con el árbol, donde Vida también aparecía entre las coberturas más seguras.
* Si la cobertura es Especiales, el riesgo también es bajo (0.39x).
* Si la cobertura es Invalidez, el riesgo es prácticamente igual (1.06x) al de referencia — no aporta información diferenciadora.
* Ciertos diagnósticos elevan fuertemente el riesgo: Dengue Hemorrágico (59.5 veces más), Dengue clásico (44.4 veces), Fractura del esternón (21.3 veces), Fiebre amarilla (19.7 veces) y COVID-19 (10.9 veces), todos comparados contra la categoría de diagnóstico menos frecuente que quedó como referencia.

**En conjunto**, la regresión y el árbol coinciden en las mismas variables y en la misma dirección del efecto (Tipo apertura, Cobertura, dias_siniestro_a_notificacion), aunque llegaron ahí por caminos completamente distintos. Que dos modelos de naturaleza tan diferente encuentren los mismos patrones es una señal fuerte de que estos hallazgos son reales y no un artefacto de un solo algoritmo.

In [0]:
import numpy as np
import pandas as pd

coeficientes = pd.DataFrame({
    'variable': X_train.columns,
    'coeficiente': best_lr.coef_[0]
})

# Convertir el coeficiente a odds ratio (mas facil de interpretar como "veces mas riesgo")
coeficientes['odds_ratio'] = np.exp(coeficientes['coeficiente'])

# Ordenar de mayor a menor riesgo
coeficientes = coeficientes.sort_values('odds_ratio', ascending=False)

print(coeficientes.to_string(index=False))

##3.CatBoostClassifier
El tercer modelo a implementar es CatBoost, un metodo de ensamble que combina muchos arboles de decision entrenados uno detras de otro, cada uno corrigiendo lo que el anterior se equivoco. Lo elijo porque maneja directamente las variables categoricas sin necesidad de transformarlas antes, y suele dar el mejor desempeño de los tres modelos.

**Desventajas:** al combinar tantos arboles, se pierde la facilidad para explicar de un vistazo por que predice lo que predice, toca apoyarse en herramientas extra para entenderlo bien. Tambien tarda mas en entrenar que los otros dos modelos.

####**Preparación de los datos**
Con CatBoost la preparacion es mucho mas simple. No hace falta convertir las categorias a numero ni crear dummies, el modelo las recibe directo tal como estan en texto.

Solo se codifico la variable objetivo (Fraude S /N) a 0 y 1, ya que eso si lo pide la libreria.

In [0]:
#creamos una copia de la base original
Base_catboost = Base_sin_duplicados.copy()

#codificamos la variable objetivo, ya que la libreria si la necesita en numero
Base_catboost['Fraude S /N'] = Base_catboost['Fraude S /N'].map({'No es fraude': 0, 'Fraude': 1})

#a diferencia del arbol y la regresion, CatBoost recibe las categoricas directo en texto
#solo hay que indicarle cuales columnas son categoricas al momento de entrenar
columnas_categoricas = ['SEXO_asegurado', 'SUCURSAL', 'Cobertura', 'Tipo apertura', 'diagnostico_agrupado', 'Rango_edad']

Base_catboost.head(5)

####**Creación del conjunto de prueba y el conjunto de entrenamiento**
Misma proporción 80-20, manteniendo el equilibrio de la variable objetivo en ambas bases. A diferencia de la regresión, aquí el número de columnas se mantiene igual de compacto que en el árbol, ya que no se necesitaron dummies para las categóricas.

In [0]:
from sklearn.model_selection import train_test_split

X = Base_catboost.drop(columns=['Fraude S /N'])
y = Base_catboost['Fraude S /N']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Proporcion de fraude en train: {y_train.mean():.2%}")
print(f"Proporcion de fraude en test: {y_test.mean():.2%}")

####**Entrenamiento del modelo**
Entrenamos nuestro CatBoost con hiperparametrización, buscando la combinación de profundidad, tasa de aprendizaje e iteraciones que mejor califica el modelo. Guardamos la semilla para que los resultados sean siempre los mismos al ejecutar de nuevo.

In [0]:
from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV

# Indices de las columnas categoricas
indices_categoricas = [X_train.columns.get_loc(col) for col in columnas_categoricas]

# Crear el modelo SIN cat_features en el constructor
cb = CatBoostClassifier(random_state=42, verbose=0)

# Definir los hiperparametros que se desean ajustar
param_grid = {
    'depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'iterations': [100, 200]
}

# Buscar la mejor combinacion, pasando cat_features como parametro del fit
grid_search = GridSearchCV(estimator=cb, param_grid=param_grid, cv=3, scoring='f1')
grid_search.fit(X_train, y_train, cat_features=indices_categoricas)

# Obtener los mejores hiperparametros
best_params = grid_search.best_params_
print("Mejores hiperparametros:", best_params)

# Entrenar el modelo final, aqui si podemos pasar cat_features directo (no hay clonacion)
best_cb = CatBoostClassifier(cat_features=indices_categoricas, random_state=42, verbose=0, **best_params)
best_cb.fit(X_train, y_train)

####**Evaluación del modelo**
CatBoost supera con claridad a los dos modelos anteriores en casi todas las métricas: detecta 9 de cada 10 casos de fraude (recall del 90%), muy por encima del 77% del árbol y el 82% de la regresión. Además, cuando marca una alarma acierta el 81% de las veces, mejor que la regresión (72%) y cerca del árbol (86%). En balance general (F1 de 0.85) y en capacidad de discriminación (AUC de 0.88) también lidera frente a los otros dos. Es, hasta ahora, el modelo más completo: detecta más fraude sin sacrificar tanta precisión como la regresión.

In [0]:
from sklearn import metrics
import matplotlib.pyplot as plt

# Evaluar el modelo
y_pred_best = best_cb.predict(X_test)

acc = metrics.accuracy_score(y_test, y_pred_best)
print(f'Exactitud: {acc}')

matconf = metrics.confusion_matrix(y_test, y_pred_best, labels=[1,0])
print(f'Matriz de Confusion (filas:real, col:prediccion): \n{matconf}')

p = metrics.precision_score(y_test, y_pred_best, labels=[1,0], average=None)
print(f'Precision: {p}')

r = metrics.recall_score(y_test, y_pred_best, labels=[1,0], average=None)
print(f'Cobertura (recall): {r}')

f1 = metrics.f1_score(y_test, y_pred_best, labels=[1,0], average=None)
print(f'F1: {f1}')

fpr, tpr, thresholds = metrics.roc_curve(y_test, y_pred_best, pos_label=1)
plt.plot(fpr, tpr)

auc = metrics.auc(fpr, tpr)
print(f'Area bajo la curva ROC: {auc}')

### Interpretación de CatBoost (feature importance)

**SUCURSAL (19.1)** sale como la variable mas importante para este modelo. Tiene sentido, ya la habiamos visto como la de mayor correlacion con el objetivo (0.67) desde el principio, solo que en la regresion se quedo repartida entre muchas dummies y no se veia tan clara. CatBoost la maneja de una sola vez y ahi si se nota su peso real.

**dias_siniestro_a_notificacion (16.4)** confirma lo mismo que ya mostraban el arbol (era la variable raiz) y la regresion (el mayor odds ratio, 11.7x). Los tres modelos coinciden: notificar tarde es de las señales mas fuertes de fraude.

**diagnostico_agrupado (15.8)** tambien coincide con lo visto antes, ya se habia notado su relacion fuerte con el fraude tanto en la matriz de correlacion como en los odds ratio de la regresion.

**dias_vigencia_hasta_siniestro (11.5)** coincide con el arbol y con la regresion: siniestros cercanos al inicio de la poliza siguen siendo señal de riesgo.

**Tipo apertura (10.3)** baja un poco en el ranking pero sigue siendo relevante, tal como en los otros dos modelos.

El resto (num_reclamaciones_previas, Rango_edad, Cobertura, SEXO_asegurado) aporta menos.

En resumen, CatBoost cambia el orden (SUCURSAL sube al primer lugar) pero las variables de fondo son las mismas en los tres modelos. Que tres algoritmos tan distintos lleguen a las mismas conclusiones le da mucha fuerza a estos hallazgos.

In [0]:
importancias = pd.DataFrame({
    'variable': X_train.columns,
    'importancia': best_cb.get_feature_importance()
}).sort_values('importancia', ascending=False)

print(importancias.to_string(index=False))

## Modelo elegido

En particular, cada modelo tiene sus ventajas y desventajas. Lo que más me gusta del árbol de decisión es que es sencillo y graficable; lo malo es que requiere variables numéricas, lo que entorpece un poco la interpretación directa de las categorías. Lo que me gusta de la regresión es que da muchas pistas de lo que lleva a la variable objetivo, con un nivel de detalle mayor que el árbol; sin embargo, no maneja bien variables con muchas categorías. Por último está CatBoost, que como era de esperarse por ser un método de ensamble presenta las mejores métricas; lo malo, como ya se mencionó, es su interpretabilidad, al ser un conjunto de árboles no es tan simple de explicar.

Si me preguntaran, para darle una visión al negocio usaría el árbol de decisión, de modo que puedan conocer con claridad las características de un perfil fraudulento. Pero si fuera el motor directo de una herramienta que solo entrega resultados a partir de la información que recibe, elegiría CatBoost, por ser el que presenta las mejores métricas.

## Conclusiones y observaciones

1. Es de suma importancia que desde el principio se tenga el **conocimiento de negocio** y **definición de las variables**, esto nos da una visión más clara del comportamiento normal o anormal de las mismas. En este caso fue clave para detectar variables con riesgo de fuga de información (estado, valores de pago y reservas), que a simple vista parecían buenos predictores pero en realidad eran consecuencia del mismo veredicto que se quería predecir.

2. El proceso de limpieza de información no es de solo una sección, se hace de manera **iterativa**, de **principio a fin**. Varias decisiones (como la normalización de categorías o la eliminación de variables redundantes) se fueron ajustando a medida que aparecía nueva evidencia en el camino.

3. Fue de gran importancia conocer el **contenido** de algunas variables antes de **descartarlas**, ya que lo que aparenta ser un formato dañado de columna puede ser información no estructurada esperando a ser transformada. El caso más claro fue *Fecha_Primer_Cierre_Siniestro*, que mezclaba fechas nativas, texto en formato día/mes/año y valores inválidos tipo "?", y que finalmente terminó descartándose por leakage, no por su formato.

4. Probablemente la razón de que el **rendimiento** de los tres modelos diera relativamente cercano en algunas métricas es la cantidad de información con la que se está trabajando. Sin embargo, CatBoost sí mostró una ventaja clara sobre los otros dos, especialmente en recall (90% frente a 76.7% del árbol y 82% de la regresión) y en el balance general F1 (0.85). El rendimiento de todos podría mejorar y las diferencias entre modelos podrían acentuarse mucho más con volúmenes de información mayores a los 12.776 registros disponibles.

5. Empezamos con **12.776 filas y 40 columnas** y terminamos con ** 11891 registros y 10 variables predictoras + 1 variable objetivo** para modelar, tras un extenso proceso de depuración de variables irrelevantes, redundantes y con riesgo de fuga de información.